<a href="https://colab.research.google.com/github/Ebad-ur-Rehman/llm-medical-qa-evaluation/blob/main/llm_medical_qa_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers accelerate bitsandbytes -q

from datasets import load_dataset

# Using a script-free version of MedQA
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

print(dataset)
print(dataset['test'][0])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 15.4 MB/s eta 0:00:00


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl: reconstructing file:   0%|          |  0.00B / 16.2MB            

phrases_no_exclude_train.jsonl: downloading bytes:           |  0.00B            

phrases_no_exclude_test.jsonl:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})
{'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?', 'answer': 'Tell the attending that he cannot fail to discl

In [3]:
import random

random.seed(42)  # ensures same sample every time you rerun

test_data = dataset['test']
sample_size = 150  # adjust if you want more/fewer questions
sample_indices = random.sample(range(len(test_data)), sample_size)
sample_questions = test_data.select(sample_indices)

print(f"Sample size: {len(sample_questions)}")
print(sample_questions[0])

Sample size: 150
{'question': 'A 21-year-old male presents to his primary care provider for fatigue. He reports that he graduated from college last month and returned 3 days ago from a 2 week vacation to Vietnam and Cambodia. For the past 2 days, he has developed a worsening headache, malaise, and pain in his hands and wrists. The patient has a past medical history of asthma managed with albuterol as needed. He is sexually active with both men and women, and he uses condoms “most of the time.” On physical exam, the patient’s temperature is 102.5°F (39.2°C), blood pressure is 112/66 mmHg, pulse is 105/min, respirations are 12/min, and oxygen saturation is 98% on room air. He has tenderness to palpation over his bilateral metacarpophalangeal joints and a maculopapular rash on his trunk and upper thighs. Tourniquet test is negative. Laboratory results are as follows:\n\nHemoglobin: 14 g/dL\nHematocrit: 44%\nLeukocyte count: 3,200/mm^3\nPlatelet count: 112,000/mm^3\n\nSerum:\nNa+: 142 mEq/

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config so it fits on the free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully


In [6]:
def format_question(example):
    question = example['question']
    options = example['options']  # dict like {'A': '...', 'B': '...', 'C': '...', 'D': '...'}

    prompt = f"""You are a medical expert answering a multiple-choice question.

Question: {question}

Options:
A) {options['A']}
B) {options['B']}
C) {options['C']}
D) {options['D']}

Respond with ONLY the letter of the correct answer (A, B, C, or D). Do not explain your reasoning.

Answer:"""
    return prompt

# Test on one example first
test_example = sample_questions[0]
prompt = format_question(test_example)
print(prompt)

You are a medical expert answering a multiple-choice question.

Question: A 21-year-old male presents to his primary care provider for fatigue. He reports that he graduated from college last month and returned 3 days ago from a 2 week vacation to Vietnam and Cambodia. For the past 2 days, he has developed a worsening headache, malaise, and pain in his hands and wrists. The patient has a past medical history of asthma managed with albuterol as needed. He is sexually active with both men and women, and he uses condoms “most of the time.” On physical exam, the patient’s temperature is 102.5°F (39.2°C), blood pressure is 112/66 mmHg, pulse is 105/min, respirations are 12/min, and oxygen saturation is 98% on room air. He has tenderness to palpation over his bilateral metacarpophalangeal joints and a maculopapular rash on his trunk and upper thighs. Tourniquet test is negative. Laboratory results are as follows:

Hemoglobin: 14 g/dL
Hematocrit: 44%
Leukocyte count: 3,200/mm^3
Platelet count:

In [9]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("Model response:", response)
print("Correct answer:", test_example['answer_idx'])

Model response: A) Chikungunya.


Correct answer: A


In [10]:
import re

def extract_answer_letter(response):
    """Extract the first standalone A/B/C/D letter from the model's response."""
    match = re.search(r'\b([ABCD])\b', response)
    if match:
        return match.group(1)
    return None  # couldn't parse - we'll count these separately

# Test it on your example
predicted = extract_answer_letter(response)
print(f"Extracted answer: {predicted}")
print(f"Correct answer: {test_example['answer_idx']}")
print(f"Match: {predicted == test_example['answer_idx']}")

Extracted answer: A
Correct answer: A
Match: True


In [11]:
import time
import json

results = []

for i, example in enumerate(sample_questions):
    prompt = format_question(example)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    predicted = extract_answer_letter(response)
    correct = example['answer_idx']
    is_correct = (predicted == correct)

    results.append({
        'question_idx': i,
        'question': example['question'][:100],  # truncated for readability
        'predicted': predicted,
        'correct': correct,
        'is_correct': is_correct,
        'raw_response': response
    })

    # Progress update every 10 questions
    if (i + 1) % 10 == 0:
        current_acc = sum(r['is_correct'] for r in results) / len(results)
        print(f"Progress: {i+1}/{len(sample_questions)} | Running accuracy: {current_acc:.2%}")

    # Safety save every 25 questions, in case runtime disconnects
    if (i + 1) % 25 == 0:
        with open('mistral_results_partial.json', 'w') as f:
            json.dump(results, f, indent=2)

# Final save
with open('mistral_results_final.json', 'w') as f:
    json.dump(results, f, indent=2)

final_accuracy = sum(r['is_correct'] for r in results) / len(results)
unparseable = sum(1 for r in results if r['predicted'] is None)

print(f"\n=== FINAL RESULTS ===")
print(f"Total questions: {len(results)}")
print(f"Accuracy: {final_accuracy:.2%}")
print(f"Unparseable responses: {unparseable}")

Progress: 10/150 | Running accuracy: 50.00%
Progress: 20/150 | Running accuracy: 50.00%
Progress: 30/150 | Running accuracy: 53.33%
Progress: 40/150 | Running accuracy: 42.50%
Progress: 50/150 | Running accuracy: 46.00%
Progress: 60/150 | Running accuracy: 46.67%
Progress: 70/150 | Running accuracy: 48.57%
Progress: 80/150 | Running accuracy: 48.75%
Progress: 90/150 | Running accuracy: 47.78%
Progress: 100/150 | Running accuracy: 50.00%
Progress: 110/150 | Running accuracy: 46.36%
Progress: 120/150 | Running accuracy: 45.83%
Progress: 130/150 | Running accuracy: 47.69%
Progress: 140/150 | Running accuracy: 47.14%
Progress: 150/150 | Running accuracy: 48.67%

=== FINAL RESULTS ===
Total questions: 150
Accuracy: 48.67%
Unparseable responses: 0
